# Purpose:
- Analyzing OMC results (after dev)
- Focusing on z-drift only for now.
    - Consider fluorescence signal later

In [4]:
from split_reg_emf import _extract_dict_from_si_string, create_plane_h5_from_tiff, register_plane_and_save_emf
from pathlib import Path
from ScanImageTiffReader import ScanImageTiffReader
import tifffile
from dask import delayed, compute
from dask.distributed import Client
from time import time
import numpy as np
import pandas as pd
import h5py

from lamf_analysis.ophys import zdrift

%load_ext autoreload
%autoreload 2

# Split, register, segment, and save them all

In [3]:
def split_reg_emf(
    data_fn: str,
    num_planes: int = 8,
    num_threads: int = 10,
    epoch_minutes: int = 1,
):
    """
    Splits a ScanImageTiffReader file into multiple h5 files and registers them to create an EMF.

    Parameters
    ----------
    data_fn : str
        Path to the input TIFF file.
    num_planes : int
        Number of planes to split the TIFF file into.
    num_threads : int
        Number of threads to use for processing.
    epoch_minutes : int
        Time in minutes for each epoch.

    Returns
    -------
    None
    """

    # get metadata from tiff file
    with ScanImageTiffReader(str(data_fn)) as reader:
        md_string = reader.metadata()

    with tifffile.TiffFile(data_fn) as tif:
        num_pages = (len(tif.pages))

    # split si & roi groups, prep for seprate parse
    s = md_string.split("\n{")
    rg_str = "{" + s[1]
    si_str = s[0]

    # parse 1: extract keys and values, dump, then load again
    si_metadata = _extract_dict_from_si_string(si_str)
    frame_rate = float(si_metadata['SI.hRoiManager.scanVolumeRate'])

    with Client() as client:
        tasks = [delayed(create_plane_h5_from_tiff)(data_fn, pi, num_pages, num_planes) for pi in range(num_planes)]
        results = compute(*tasks, num_workers=num_threads)

        tasks = [delayed(register_plane_and_save_emf)(h5_fn, frame_rate, epoch_minutes) for h5_fn in results]
        compute(*tasks, num_workers=num_threads)


In [5]:
# this takes too long.
# use HPC (15 min session takes about 6 minutes)
data_dir = Path(r'\\allen\programs\mindscope\workgroups\learning\pilots\online_motion_correction\250423_783551')
filenames = ['1433053813_timeseries_00013.tif',
            '1433053813_timeseries_00014.tif',
]

for fn in filenames:
    t0 = time()
    data_fn = data_dir / fn
    print(f'Processing {data_fn}')
    split_reg_emf(
        data_fn,
        num_planes=8,
        num_threads=8,
        epoch_minutes=1,
    )
    t1 = time()
    print(f'Finished {data_fn} in {t1-t0:.2f} seconds')

Processing \\allen\programs\mindscope\workgroups\learning\pilots\online_motion_correction\250423_783551\1433053813_timeseries_00013.tif
Finished \\allen\programs\mindscope\workgroups\learning\pilots\online_motion_correction\250423_783551\1433053813_timeseries_00013.tif in 2952.71 seconds
Processing \\allen\programs\mindscope\workgroups\learning\pilots\online_motion_correction\250423_783551\1433053813_timeseries_00014.tif
Finished \\allen\programs\mindscope\workgroups\learning\pilots\online_motion_correction\250423_783551\1433053813_timeseries_00014.tif in 1584.89 seconds


## how to use HPC
- sbatch_split_reg_emf_slurm.py
- change parameters (directory, num_planes, epoch duration)

# z-drift calculation
- There is HPC code, but it needs modification
    - File formats changed. Also we can depend on the file naming convention.
        - It will be faster and more reliable.
    - Try this for 2 calculations
        1. Z-drift and correlation calculation during OMC (from single frames, using the information in the csv file)
            - To test if we can improve estimation algorithm, especially for low SNR images (deepest plane)
        2. Z-drift calculation using EMF.
    - Compare this with the estimation and correction log.
        - We don't have the correction log yet (04/25/2025)
    


In [5]:
data_dir = Path(r'\\allen\programs\mindscope\workgroups\learning\pilots\online_motion_correction\250423_783551')
zstack_dir = data_dir / 'sorted_local_z_stacks'
file_inds = [13, 14] # index of files to process
filenames_all = data_dir.glob('*.tif*')
session_names = np.unique([f.stem.split('_')[0] for f in filenames_all])
assert len(session_names) == 1
session_name = int(session_names[0])

## 1. z-drift and correlation calculation during OMC (from single frames)

In [6]:
file_ind = file_inds[0]
reg_filenames = list(data_dir.glob(f'{session_name}_timeseries_{file_ind:05}_*_reg.h5'))
num_planes = len(reg_filenames)
# check filename format
reg_filenames_confirm = [data_dir / f'{session_name}_timeseries_{file_ind:05}_{pi:02}_reg.h5' for pi in range(num_planes)]
assert reg_filenames == reg_filenames_confirm, f'Filename format mismatch: {reg_filenames} != {reg_filenames_confirm}'

emf_filenames = [fn.parent / f'{fn.name.split("_reg")[0]}_emf.h5' for fn in reg_filenames]
for fn in emf_filenames:
    assert fn.exists()

In [21]:
FOV_ORDER_DICT = {
    0: '0_reg_ch_1',
    1: '0_reg_ch_2',
    2: '1_reg_ch_1',
    3: '1_reg_ch_2',
    4: '2_reg_ch_1',
    5: '2_reg_ch_2',
    6: '3_reg_ch_1',
    7: '3_reg_ch_2',
}

def get_zstack(zstack_dir, plane_ind):
    matched_zstack_key = FOV_ORDER_DICT[plane_ind]
    matched_zstack_fn = list(zstack_dir.glob(f'*_local_z_stack{matched_zstack_key}.tif'))
    assert len(matched_zstack_fn) == 1
    matched_zstack_fn = matched_zstack_fn[0]
    zstack = tifffile.imread(matched_zstack_fn)
    return zstack


def get_motion_range(ops_fn):
    ops = np.load(ops_fn, allow_pickle=True).item()
    y_offs = ops['reg_result'][4][0]
    x_offs = ops['reg_result'][4][1]
    assert max(y_offs) > 0
    assert min(y_offs) < 0
    assert max(x_offs) > 0
    assert min(x_offs) < 0
    range_y = [max(y_offs), min(y_offs)]
    range_x = [max(x_offs), min(x_offs)]
    return range_y, range_x


def calculate_zdrift(zstack, emf, range_y, range_x,
                     use_clahe=True, use_valid_pix=True):
    # prepare zstack and crop emf
    zstack_crop = zstack[:, range_y[0]:range_y[1], range_x[0]:range_x[1]]
    stack_pre = zdrift.med_filt_z_stack(zstack_crop)
    stack_pre = zdrift.rolling_average_stack(stack_pre)    
    episodic_mean_fovs_crop = emf[:, range_y[0]:range_y[1], range_x[0]:range_x[1]]

    # calculate
    matched_plane_indices = np.zeros(
    episodic_mean_fovs_crop.shape[0], dtype=int)
    corrcoef = []
    segment_reg_imgs = []
    shift_list = []
    for i in range(episodic_mean_fovs_crop.shape[0]):
        fov_reg_stack, cc, shift = zdrift.fov_stack_register_phase_correlation(
            episodic_mean_fovs_crop[i], stack_pre, use_clahe=use_clahe,
            use_valid_pix=use_valid_pix)
        matched_plane_indices[i] = np.argmax(cc)
        corrcoef.append(cc)
        segment_reg_imgs.append(fov_reg_stack[np.argmax(cc)])
        shift_list.append(shift)
    corrcoef = np.asarray(corrcoef)

    results = {'matched_plane_indices': matched_plane_indices,
                'corrcoef': corrcoef,
                'segment_fov_registered': segment_reg_imgs,
                'ref_zstack_crop': ref_zstack_crop,                   
                'shift': shift_list,
                'use_clahe': use_clahe,
                'use_valid_pix': use_valid_pix}
    return results

In [23]:
# takes about 3 minutes per plane (of 15 minutes session)
num_planes = 8
reg_filenames = list(data_dir.glob(f'{session_name}_timeseries_{file_ind:05}_*_reg.h5'))
assert num_planes == len(reg_filenames)
# check filename format
reg_filenames_confirm = [data_dir / f'{session_name}_timeseries_{file_ind:05}_{pi:02}_reg.h5' for pi in range(num_planes)]
assert reg_filenames == reg_filenames_confirm, f'Filename format mismatch: {reg_filenames} != {reg_filenames_confirm}'

emf_filenames = [fn.parent / f'{fn.name.split("_reg")[0]}_emf.h5' for fn in reg_filenames]
ops_filenames = [fn.parent / f'{fn.name.split("_reg")[0]}_ops.npy' for fn in reg_filenames]
assert np.all([fn.exists() for fn in emf_filenames]), f'Not all emf files exist: {emf_filenames}'
assert np.all([fn.exists() for fn in ops_filenames]), f'Not all ops files exist: {ops_filenames}'

results_session = []
for plane_ind in range(num_planes):
    emf_fn = emf_filenames[plane_ind]
    ops_fn = ops_filenames[plane_ind]
    with h5py.File(emf_fn, 'r') as f:
        emf = f['data'][:]
    y_range, x_range = get_motion_range(ops_fn)
    
    results = calculate_zdrift(zstack, emf, y_range, x_range)
    results['plane_ind'] = plane_ind
    results['emf_fn'] = emf_fn
    results['ops_fn'] = ops_fn
    results['zstack_fn'] = matched_zstack_fn
    results_session.append(results)
        

In [32]:
results_session[4]['matched_plane_indices']

array([79, 78, 79, 79, 79, 79, 79, 79, 79, 79, 79, 79, 79, 79, 79, 79])

In [25]:
results_session[0]

{'matched_plane_indices': array([43, 42, 41, 42, 42, 43, 41, 41, 41, 41, 42, 42, 41, 41, 42, 42]),
 'corrcoef': array([[0.31041429, 0.31078206, 0.31397691, ..., 0.48088898, 0.47444206,
         0.47001404],
        [0.33927327, 0.33996422, 0.34333971, ..., 0.50591419, 0.50176593,
         0.49717591],
        [0.34078132, 0.34199555, 0.34454783, ..., 0.49849476, 0.49392519,
         0.48913805],
        ...,
        [0.32930661, 0.32900811, 0.33156794, ..., 0.47194519, 0.46478444,
         0.45988256],
        [0.32504616, 0.32361616, 0.32592076, ..., 0.4816803 , 0.47733537,
         0.47241311],
        [0.3296324 , 0.3288719 , 0.33106661, ..., 0.48255286, 0.47523234,
         0.46756306]]),
 'segment_fov_registered': [array([[  0.        ,   0.        ,   0.        , ...,   0.        ,
            0.        ,   0.        ],
         [  0.        ,   0.        ,   0.        , ...,   0.        ,
            0.        ,   0.        ],
         [  0.        ,   0.        ,   0.        , 

In [8]:
def reformat_df(df):
    # strip blank spaces from column names
    df.columns = df.columns.str.strip()
    for col in df.columns:
        if isinstance(df[col].values[0], str):
            df[col] = df[col].str.strip()
    df['roiName'] = df['roiName'].str.strip()
    # convert str of list to a list
    def _str_to_float_list(x):
        """
        Convert a string representation of a list to a list of floats.
        """
        return [float(np.round(float(s), 4)) for s in x.split('[')[1].split(']')[0].split(' ')]
    list_columns = ['drPixel', 'drRef', 'confidence']
    for col in list_columns:
        df[col] = df[col].apply(lambda x: _str_to_float_list(x))
    return df

motion_est_fn = data_dir / f'{session_name}_timeseries_Motion_{file_ind:05}.csv'
motion_est = pd.read_csv(motion_est_fn)
motion_est = reformat_df(motion_est)


In [9]:
plane_ind = 0

# get emf
emf_fn = emf_filenames[plane_ind]
with h5py.File(emf_fn, 'r') as h:
    emf = h['data'][:]

# get z-stack
matched_zstack_key = fov_order_dict[plane_ind]
matched_zstack_fn = list(zstack_dir.glob(f'*_local_z_stack{matched_zstack_key}.tif'))
assert len(matched_zstack_fn) == 1
matched_zstack_fn = matched_zstack_fn[0]
zstack = tifffile.imread(matched_zstack_fn)

# get ops
ops_fn = emf_fn.parent / f'{emf_fn.name.split("_emf")[0]}_ops.npy'